# Chess Learning â€” Stage 02: Self-Play & TD Learning

This notebook implements the training loop:

| Component | Description |
|---|---|
| **Self-play loop** | Engine plays both sides; positions and outcomes are recorded |
| **TD(0) learning** | Each position's target is the evaluator's own next-position prediction (negated for perspective), not the final game outcome |
| **SQLite logging** | Weight snapshots and game stats are written to a database for analysis in R |

**Key difference from Monte Carlo (`selfplay_mc.ipynb`):** In Monte Carlo, every position in a game gets the same target (the final outcome), causing large accumulated errors and draw erosion. In TD(0), the target for position *t* is `-V(s_{t+1})` — the evaluator's own prediction one step ahead, negated for the perspective flip. Only the terminal position uses the actual game outcome. This naturally limits how far the draw signal propagates.

**Design choices:**
- Default depth: 2 (~3.8s/game; ~9,500 games per 10-hour run)
- Move limit: 200 half-moves per game
- Weight checkpoints every 10 games
- Per-game mean feature values logged to `feature_stats` for feature-outcome correlation analysis


## Setup

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'chess', '-q'])

0

In [2]:
import chess
import chess.pgn
import io
import random
import sqlite3
import time
import os
from datetime import datetime

### Import engine components

In **Positron**, `engine.py` is imported directly from the same folder.
In **Google Colab**, upload `engine.py` to the session first (Files â†’ Upload), then this import will work.


In [3]:
sys.path.insert(0, os.path.dirname(os.path.abspath('selfplay.ipynb')))
from engine import FEATURE_NAMES, default_weights, extract_features, get_best_move
print('Engine loaded. Features:', FEATURE_NAMES)

Engine loaded. Features: ['passed_pawn', 'doubled_pawn', 'isolated_pawn', 'king_safety', 'center_control', 'rook_open_file', 'connected_rooks', 'bishop_pair', 'rook_seventh', 'piece_development', 'mobility', 'pawn_advancement']


## Database Setup

Nine tables:
- **`runs`** â€” one row per training run (hyperparameters, timestamp)
- **`games`** â€” one row per game (outcome, length, duration, termination reason)
- **`weights`** â€” weight snapshots every `checkpoint_every` games (primary research data)
- **`feature_stats`** â€” per-game mean feature values (white's perspective) for feature-outcome correlation analysis
- **`game_records`** â€” full PGN and mean TD error for every game; enables replay and identification of high-impact games
- **`weight_deltas`** â€” per-feature weight change from each game's TD update; use to find which games caused large weight jumps
- **`position_logs`** â€” per-ply feature vectors for every position in every game; optional (controlled by `LOG_POSITIONS`)


In [4]:
def setup_database(db_path):
    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.executescript('''
        CREATE TABLE IF NOT EXISTS runs (
            run_id        INTEGER PRIMARY KEY,
            started_at    TEXT,
            depth         INTEGER,
            learning_rate REAL,
            move_limit    INTEGER,
            notes         TEXT
        );
        CREATE TABLE IF NOT EXISTS games (
            game_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            n_halfmoves   INTEGER,
            duration_s    REAL,
            terminated_by TEXT
        );
        CREATE TABLE IF NOT EXISTS weights (
            weight_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            weight_value  REAL
        );
        CREATE TABLE IF NOT EXISTS feature_stats (
            stat_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            feature_name  TEXT,
            mean_value    REAL
        );
        CREATE TABLE IF NOT EXISTS game_records (
            record_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            pgn           TEXT,
            td_error      REAL
        );
        CREATE TABLE IF NOT EXISTS weight_deltas (
            delta_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            delta         REAL
        );
        CREATE TABLE IF NOT EXISTS position_logs (
            pos_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            ply           INTEGER,
            color         TEXT,
            feature_name  TEXT,
            feature_value REAL
        );
        CREATE TABLE IF NOT EXISTS baseline_evals (
            eval_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id       INTEGER,
            game_number  INTEGER,
            n_games      INTEGER,
            wins         INTEGER,
            losses       INTEGER,
            draws        INTEGER,
            win_rate     REAL
        );
        CREATE TABLE IF NOT EXISTS stockfish_evals (
            eval_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id       INTEGER,
            game_number  INTEGER,
            sf_elo       INTEGER,
            n_games      INTEGER,
            wins         INTEGER,
            losses       INTEGER,
            draws        INTEGER,
            win_rate     REAL,
            source       TEXT
        );
    ''')
    conn.commit()
    # Safe schema migration: add last_updated column if not already present
    try:
        conn.execute('ALTER TABLE runs ADD COLUMN last_updated TEXT')
        conn.commit()
    except sqlite3.OperationalError:
        pass  # column already exists
    return conn


def log_run(conn, run_id, depth, learning_rate, move_limit, notes=''):
    conn.execute(
        'INSERT OR REPLACE INTO runs (run_id, started_at, depth, learning_rate, move_limit, notes) '
        'VALUES (?, ?, ?, ?, ?, ?)',
        (run_id, datetime.now().isoformat(), depth, learning_rate, move_limit, notes)
    )
    conn.commit()


def log_game(conn, run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by):
    conn.execute(
        'INSERT INTO games (run_id,game_number,outcome,n_halfmoves,duration_s,terminated_by) '
        'VALUES (?,?,?,?,?,?)',
        (run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by)
    )
    conn.commit()


def log_weights(conn, run_id, game_number, weights):
    rows = [(run_id, game_number, name, val) for name, val in weights.items()]
    conn.executemany(
        'INSERT INTO weights (run_id,game_number,feature_name,weight_value) VALUES (?,?,?,?)',
        rows
    )
    conn.commit()


def log_feature_stats(conn, run_id, game_number, positions, outcome):
    # Compute mean of each feature across all positions, converted to white's perspective.
    # White-turn positions: use value as-is (positive = good for white).
    # Black-turn positions: negate (feature is from black's perspective, flip for white).
    # This puts all games on a common scale for feature-outcome correlation analysis.
    if not positions:
        return
    sums = {name: 0.0 for name in FEATURE_NAMES}
    for features, color in positions:
        sign = 1.0 if color == chess.WHITE else -1.0
        for name, val in features.items():
            sums[name] += sign * val
    n = len(positions)
    rows = [(run_id, game_number, outcome, name, sums[name] / n) for name in FEATURE_NAMES]
    conn.executemany(
        'INSERT INTO feature_stats (run_id,game_number,outcome,feature_name,mean_value) VALUES (?,?,?,?,?)',
        rows
    )
    conn.commit()


def load_last_weights(conn, run_id):
    # Load weights from the most recent checkpoint for a given run_id.
    max_game = conn.execute(
        'SELECT MAX(game_number) FROM weights WHERE run_id = ?', (run_id,)
    ).fetchone()[0]
    if max_game is None:
        raise ValueError(f'No weight checkpoints found for run_id={run_id}')
    rows = conn.execute(
        'SELECT feature_name, weight_value FROM weights '
        'WHERE run_id = ? AND game_number = ?',
        (run_id, max_game)
    ).fetchall()
    return {name: val for name, val in rows}


def get_last_game_number(conn, run_id):
    # Return the highest game_number logged for a given run_id (0 if none).
    result = conn.execute(
        'SELECT MAX(game_number) FROM games WHERE run_id = ?', (run_id,)
    ).fetchone()[0]
    return result if result is not None else 0


def log_game_record(conn, run_id, game_number, pgn, td_error):
    conn.execute(
        'INSERT INTO game_records (run_id,game_number,pgn,td_error) VALUES (?,?,?,?)',
        (run_id, game_number, pgn, td_error)
    )
    conn.commit()


def log_weight_deltas(conn, run_id, game_number, deltas):
    rows = [(run_id, game_number, name, delta) for name, delta in deltas.items()]
    conn.executemany(
        'INSERT INTO weight_deltas (run_id,game_number,feature_name,delta) VALUES (?,?,?,?)',
        rows
    )
    conn.commit()


def log_position_logs(conn, run_id, game_number, positions):
    rows = []
    for ply, (features, color) in enumerate(positions):
        color_str = 'white' if color == chess.WHITE else 'black'
        for name, val in features.items():
            rows.append((run_id, game_number, ply, color_str, name, val))
    conn.executemany(
        'INSERT INTO position_logs (run_id,game_number,ply,color,feature_name,feature_value) '
        'VALUES (?,?,?,?,?,?)',
        rows
    )
    conn.commit()


def log_baseline_eval(conn, run_id, game_number, n_games, wins, losses, draws):
    win_rate = wins / n_games if n_games > 0 else 0.0
    conn.execute(
        'INSERT INTO baseline_evals (run_id,game_number,n_games,wins,losses,draws,win_rate) '
        'VALUES (?,?,?,?,?,?,?)',
        (run_id, game_number, n_games, wins, losses, draws, win_rate)
    )
    conn.commit()

## Self-Play Game Function

Each call plays one complete game. For each half-move, the board features and side-to-move are recorded so the TD update can process every position.

Termination conditions (in priority order):
1. Natural game end (checkmate, stalemate, insufficient material)
2. Hard move limit (safety net for very long games with near-zero weights)


In [5]:
def play_game(weights, depth, move_limit=200, epsilon=0.1):
    board = chess.Board()
    game  = chess.pgn.Game()
    node  = game
    positions     = []       # list of (features_dict, color_to_move)
    terminated_by = 'natural'

    while True:
        if board.is_game_over():
            terminated_by = 'natural'
            break
        if len(positions) >= move_limit:
            terminated_by = 'move_limit'
            break

        features = extract_features(board)
        color    = board.turn

        if random.random() < epsilon:
            move = random.choice(list(board.legal_moves))
        else:
            move, _ = get_best_move(board, weights, depth=depth, randomize=True)
        if move is None:
            break

        node = node.add_variation(move)
        board.push(move)
        positions.append((features, color))

    # Resolve outcome
    if terminated_by == 'move_limit':
        outcome = '1/2-1/2'
    else:
        outcome = board.result()

    game.headers['Result'] = outcome
    pgn_str = str(game)

    return positions, outcome, terminated_by, pgn_str

## Baseline Evaluation

`play_vs_random()` plays the current engine against a random-move opponent, alternating sides across games to control for the first-mover advantage. Results are logged to `baseline_evals` every `baseline_every` training games.

The random mover picks uniformly from all legal moves — the absolute floor of chess play. Win rate against a random mover should rise monotonically if the engine is genuinely learning.


In [6]:
def play_vs_random(weights, depth, n_games=20, move_limit=200):
    """
    Evaluate current weights against a random-move opponent.
    Engine alternates sides: white for even-indexed games, black for odd.
    No epsilon-greedy exploration -- engine plays its best moves.
    Returns (wins, losses, draws) from the engine's perspective.
    """
    wins = losses = draws = 0
    for i in range(n_games):
        board        = chess.Board()
        engine_color = chess.WHITE if i % 2 == 0 else chess.BLACK
        n_halfmoves  = 0
        while not board.is_game_over() and n_halfmoves < move_limit:
            if board.turn == engine_color:
                move, _ = get_best_move(board, weights, depth=depth, randomize=False)
            else:
                move = random.choice(list(board.legal_moves))
            if move is None:
                break
            board.push(move)
            n_halfmoves += 1
        result = board.result()
        if   result == '1-0':
            engine_won = (engine_color == chess.WHITE)
        elif result == '0-1':
            engine_won = (engine_color == chess.BLACK)
        else:
            engine_won = None
        if   engine_won is True:  wins   += 1
        elif engine_won is False: losses += 1
        else:                     draws  += 1
    return wins, losses, draws

## TD(0) Learning

After each game, weights are updated position by position. The target for each non-terminal position is the evaluator's own prediction at the *next* position, negated for the perspective flip between alternating sides. Only the terminal position uses the actual game outcome.

$$w_i \leftarrow w_i + lpha \cdot (	ext{target}_t - V(s_t)) \cdot x_i$$

where:
- $$V(s_t) = \mathbf{w} \cdot \mathbf{x}_t$$ — evaluator prediction at position *t*
- $$	ext{target}_t = -V(s_{t+1})$$ for non-terminal positions (negated: a good position for the opponent is bad for us)
- $$	ext{target}_T = 	ext{outcome}$$ for the terminal position (`+1` win, `-1` loss, `0` draw)

This eliminates large accumulated errors from Monte Carlo: a drawn game produces a target of 0 only at the terminal position, not for every position in the game.


In [7]:
def outcome_for_color(outcome, color):
    if outcome == '1-0':
        return  1.0 if color == chess.WHITE else -1.0
    elif outcome == '0-1':
        return -1.0 if color == chess.WHITE else  1.0
    return 0.0  # draw or unknown


def td_update(weights, positions, outcome, learning_rate, frozen_features=None, draw_lr_scales=None):
    frozen    = frozen_features or set()
    is_draw   = (outcome == '1/2-1/2')
    dr_scales = draw_lr_scales or {}
    """
    TD(0) update: target for each position is the evaluator's own prediction
    at the next position (negated for perspective flip), except the terminal
    position which uses the actual game outcome.
    """
    # Pre-compute V(s_t) for every position using current weights
    values = [
        sum(weights.get(name, 0.0) * val for name, val in features.items())
        for features, color in positions
    ]

    weights_before = dict(weights)
    total_error    = 0.0

    for t, (features, color) in enumerate(positions):
        if t < len(positions) - 1:
            # Non-terminal: target is negated next-position value (perspective flip)
            target = -values[t + 1]
        else:
            # Terminal: use actual game outcome for the side to move
            target = outcome_for_color(outcome, color)

        error = target - values[t]
        total_error += abs(error)
        for name, val in features.items():
            if name not in frozen:
                effective_lr = learning_rate * dr_scales.get(name, 1.0) if is_draw else learning_rate
                weights[name] += effective_lr * error * val

    # Clip weights to prevent blow-up
    for name in weights:
        weights[name] = max(-50.0, min(50.0, weights[name]))

    deltas   = {name: weights[name] - weights_before[name] for name in weights}
    td_error = total_error / len(positions) if positions else 0.0
    return deltas, td_error


## Training Loop

The training loop:
1. Plays a game
2. Updates weights via TD learning
3. Logs the game result and stats
4. Checkpoints weights to SQLite every `checkpoint_every` games

**Resume mode** (`resume=True`): loads the last saved weights for `run_id` and continues game numbering from where the previous session left off. All games remain under the same `run_id`, so the weight trajectory is a single continuous series in the database.

Weight checkpoints at game 0 (initial) and at every checkpoint interval are what the R visualization notebook reads.

In [8]:
def train(
    run_id,
    n_games,
    depth            = 2,
    learning_rate    = 0.001,
    move_limit       = 200,
    checkpoint_every = 10,
    db_path          = '../../data/chess_learning.db',
    notes            = '',
    resume           = False,
    log_positions    = True,
    weight_init      = 'random',
    epsilon          = 0.1,
    frozen_features  = None,
    draw_lr_scales   = None,
    verbose          = True,
    baseline_every   = 100,
    baseline_n_games = 20,
):
    conn = setup_database(db_path)

    if resume:
        weights    = load_last_weights(conn, run_id)
        start_game = get_last_game_number(conn, run_id) + 1
        conn.execute(
            'UPDATE runs SET last_updated = ? WHERE run_id = ?',
            (datetime.now().isoformat(), run_id)
        )
        conn.commit()
    else:
        weights    = default_weights(weight_init)
        start_game = 1
        log_run(conn, run_id, depth, learning_rate, move_limit, notes)
        log_weights(conn, run_id, 0, weights)  # checkpoint initial weights

    outcomes = {'1-0': 0, '0-1': 0, '1/2-1/2': 0}
    t_start  = time.time()

    for i in range(n_games):
        game_num = start_game + i
        t0       = time.time()
        positions, outcome, terminated_by, pgn = play_game(weights, depth, move_limit, epsilon=epsilon)
        elapsed  = time.time() - t0

        deltas, td_error = td_update(weights, positions, outcome, learning_rate, frozen_features, draw_lr_scales)
        log_game(conn, run_id, game_num, outcome, len(positions), elapsed, terminated_by)
        log_game_record(conn, run_id, game_num, pgn, td_error)
        log_weight_deltas(conn, run_id, game_num, deltas)
        log_feature_stats(conn, run_id, game_num, positions, outcome)
        if log_positions:
            log_position_logs(conn, run_id, game_num, positions)
        outcomes[outcome] = outcomes.get(outcome, 0) + 1

        if game_num % checkpoint_every == 0:
            log_weights(conn, run_id, game_num, weights)
            if verbose:
                elapsed_total = time.time() - t_start
                rate = (i + 1) / elapsed_total * 3600
                print(
                    f'Game {game_num:5d} | {outcome:7s} | {len(positions):3d} half-moves '
                    f'| {elapsed:.1f}s | td_err={td_error:.3f} '
                    f'| {rate:.0f} games/hr'
                )

        if baseline_every and game_num % baseline_every == 0:
            b_wins, b_losses, b_draws = play_vs_random(
                weights, depth, n_games=baseline_n_games, move_limit=move_limit
            )
            log_baseline_eval(conn, run_id, game_num, baseline_n_games, b_wins, b_losses, b_draws)
            if verbose:
                b_winrate = b_wins / baseline_n_games * 100
                print(
                    f'  → Baseline vs random ({baseline_n_games} games): '
                    f'{b_wins}W / {b_losses}L / {b_draws}D  '
                    f'win_rate={b_winrate:.0f}%'
                )

    final_game = start_game + n_games - 1
    log_weights(conn, run_id, final_game, weights)  # final checkpoint
    conn.close()

    total_time = time.time() - t_start
    if verbose:
        action = 'resumed' if resume else 'complete'
        print(f'\nRun {run_id} {action}: {n_games} games in {total_time/60:.1f} min '
              f'(games {start_game} to {final_game})')
        print(f'Outcomes: {outcomes}')
        print(f'Database: {os.path.abspath(db_path)}')

    return weights

## Configuration & Run

Edit the values below to configure a training session. Each run gets a unique `RUN_ID`; multiple sessions can contribute to the same run by setting `RESUME = True`.

**Typical session sizes:**
- Quick check / first session: `N_GAMES = 100` (~15â€“20 min)
- Medium session: `N_GAMES = 500` (~60â€“90 min)

**Workflow:**
1. Set `RESUME = False` and run to start a new run.
2. Examine results in R (`03_visualize.qmd`).
3. Set `RESUME = True` and run again with the same `RUN_ID` to continue.

> **Google Colab path**: set `DB_PATH` to a location on your mounted Drive,  
> e.g. `'/content/drive/MyDrive/chess_learning.db'`


In [9]:
RUN_ID           = 14
N_GAMES          = 400        # games to play in this session
DEPTH            = 3
LEARNING_RATE    = 0.001
MOVE_LIMIT       = 200       # half-moves; safety cap for early training
CHECKPOINT_EVERY = 10        # log weights to DB every N games
DB_PATH          = '../../data/chess_learning.db'
WEIGHT_INIT      = 'zero'    # 'random' or 'zero'
RESUME           = False     # set True to continue an existing run from where it left off
LOG_POSITIONS    = True      # log per-ply feature vectors; disable to save space on long runs
EPSILON          = 0.1       # epsilon-greedy exploration rate (0 = pure greedy)
DRAW_LR_SCALES   = {'king_safety': 0.05, 'center_control': 0.2}  # per-feature draw LR scale
BASELINE_EVERY   = None       # evaluate vs random mover every N training games (None to disable)
BASELINE_N_GAMES = 20        # games per baseline evaluation

final_weights = train(
    run_id           = RUN_ID,
    n_games          = N_GAMES,
    depth            = DEPTH,
    learning_rate    = LEARNING_RATE,
    move_limit       = MOVE_LIMIT,
    checkpoint_every = CHECKPOINT_EVERY,
    db_path          = DB_PATH,
    resume           = RESUME,
    log_positions    = LOG_POSITIONS,
    weight_init      = WEIGHT_INIT,
    epsilon          = EPSILON,
    draw_lr_scales   = DRAW_LR_SCALES,
    baseline_every   = BASELINE_EVERY,
    baseline_n_games = BASELINE_N_GAMES,
    notes            = f'TD(0); 14 features; hard-coded material; draw_lr king_safety=0.05 center_control=0.2; depth={DEPTH}, lr={LEARNING_RATE}, init={WEIGHT_INIT}, eps={EPSILON}',
)

Game   710 | 1-0     |  63 half-moves | 45.7s | td_err=0.055 | 58 games/hr
Game   720 | 1-0     |   7 half-moves | 2.8s | td_err=0.161 | 59 games/hr
Game   730 | 0-1     | 132 half-moves | 61.9s | td_err=0.052 | 56 games/hr
Game   740 | 1/2-1/2 | 185 half-moves | 58.7s | td_err=0.042 | 59 games/hr
Game   750 | 1/2-1/2 | 200 half-moves | 58.2s | td_err=0.044 | 60 games/hr
Game   760 | 0-1     |  58 half-moves | 29.3s | td_err=0.057 | 59 games/hr
Game   770 | 1-0     |  69 half-moves | 50.7s | td_err=0.058 | 59 games/hr
Game   780 | 1-0     | 175 half-moves | 37.6s | td_err=0.049 | 60 games/hr
Game   790 | 0-1     | 102 half-moves | 93.2s | td_err=0.056 | 60 games/hr
Game   800 | 1-0     |  55 half-moves | 39.8s | td_err=0.045 | 59 games/hr
Game   810 | 1-0     | 145 half-moves | 75.2s | td_err=0.048 | 58 games/hr
Game   820 | 0-1     |  56 half-moves | 32.9s | td_err=0.064 | 58 games/hr
Game   830 | 1-0     |  93 half-moves | 82.1s | td_err=0.059 | 58 games/hr
Game   840 | 0-1     |  70

## Weight Trajectory

Quick check: read weight snapshots back from the database and display how each feature weight evolved over the run. Richer visualization will be built in Stage 03 (R).


In [10]:
conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    'SELECT game_number, feature_name, weight_value FROM weights '
    'WHERE run_id = ? ORDER BY game_number, feature_name',
    (RUN_ID,)
).fetchall()
conn.close()

# Pivot to game_number -> feature -> value
from collections import defaultdict
by_game = defaultdict(dict)
for game_num, feature, val in rows:
    by_game[game_num][feature] = val

game_nums = sorted(by_game.keys())

header = f"{'Feature':<20}" + ''.join(f'{g:>10}' for g in game_nums)
print(header)
print('-' * len(header))
for feature in FEATURE_NAMES:
    row = f'{feature:<20}' + ''.join(f'{by_game[g].get(feature, 0):>10.4f}' for g in game_nums)
    print(row)

Feature                      0        10        20        30        40        50        60        70        80        90       100       110       120       130       140       150       160       170       180       190       200       210       220       230       240       250       260       270       280       290       300       310       320       330       340       350       360       370       380       390       400       410       420       430       440       450       460       470       480       490       500       510       520       530       540       550       560       570       580       590       600       610       620       630       640       650       660       670       680       690       700       710       720       730       740       750       760       770       780       790       800       810       820       830       840       850       860       870       880       890       900       910       920       930       940       950       960       970

---
If weights are moving (even slightly), the pipeline is working. Pronounced movement in `material_*` weights is expected first â€” this is the engine beginning to value pieces correctly. Weight trajectories will be visualized fully in Stage 03 (R).